In [1]:
# ==================================================================================
# Grad-CAM Explainability Analysis - ResNet50 Model
# ==================================================================================
# Purpose: Generate Gradient-weighted Class Activation Maps (Grad-CAM) to visualize
#          which regions of brain MRI images the ResNet50 model focuses on when 
#          making predictions. Critical for validating model attention on tumor regions.
#
# Output: 3 separate images per example (original, heatmap, overlay) for tumor and 
#         no-tumor cases with prediction confidence scores.
# ==================================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
import random

# ==================== Configuration ====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

IMG_SIZE = 224
TEST_DIR = "../dataset_processed/test"
RANDOM_SEED = 42

# ==================== Load ResNet50 Model ====================
print("\nLoading ResNet50 model...")

# Initialize architecture
resnet50 = models.resnet50(weights=None)
num_features = resnet50.fc.in_features
resnet50.fc = nn.Linear(num_features, 2)  # Binary: tumor (1) vs no tumor (0)
resnet50 = resnet50.to(device)

# Load trained weights
checkpoint = torch.load("../resnet/resnet50_best.pth", map_location=device)
resnet50.load_state_dict(checkpoint)
resnet50.eval()  # Set to evaluation mode

print(f"✓ ResNet50 loaded successfully ({sum(p.numel() for p in resnet50.parameters()):,} parameters)")

# ==================== Grad-CAM Implementation ====================
class GradCAM:
    """
    Gradient-weighted Class Activation Mapping (Grad-CAM)
    Highlights important regions in the image that influence the model's prediction.
    """
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Register hooks to capture gradients and activations
        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_backward_hook(self.save_gradient)
    
    def save_activation(self, module, input, output):
        """Hook to save forward pass activations"""
        self.activations = output.detach()
    
    def save_gradient(self, module, grad_input, grad_output):
        """Hook to save backward pass gradients"""
        self.gradients = grad_output[0].detach()
    
    def generate_heatmap(self, input_image, class_idx=None):
        """Generate Grad-CAM heatmap for the predicted or specified class"""
        # Forward pass to get prediction
        output = self.model(input_image)
        
        # Use predicted class if not specified
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()
        
        # Backward pass to get gradients
        self.model.zero_grad()
        class_score = output[:, class_idx]
        class_score.backward()
        
        # Weight activations by average gradient (global average pooling)
        pooled_gradients = torch.mean(self.gradients, dim=[0, 2, 3])
        
        # Apply weights to activations
        for i in range(self.activations.shape[1]):
            self.activations[:, i, :, :] *= pooled_gradients[i]
        
        # Average across channels to get final heatmap
        heatmap = torch.mean(self.activations, dim=1).squeeze()
        
        # Apply ReLU to focus on positive influences
        heatmap = F.relu(heatmap)
        
        # Normalize to [0, 1] range
        heatmap /= torch.max(heatmap) if torch.max(heatmap) > 0 else 1.0
        
        return heatmap.cpu().numpy(), class_idx

# ==================== Image Processing Functions ====================
def load_and_preprocess_image(img_path):
    """Load image and apply ResNet50 preprocessing (ImageNet normalization)"""
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # ImageNet stats
    ])
    
    img = Image.open(img_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0)  # Add batch dimension
    
    return img_tensor, img

def create_overlay(img_path, heatmap, alpha=0.4):
    """Overlay Grad-CAM heatmap on original image"""
    # Load original image
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    
    # Resize heatmap to image size
    heatmap_resized = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))
    
    # Convert heatmap to colored version (jet colormap)
    heatmap_colored = np.uint8(255 * heatmap_resized)
    heatmap_colored = cv2.applyColorMap(heatmap_colored, cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    
    # Blend original image with heatmap
    overlay = cv2.addWeighted(img, 1-alpha, heatmap_colored, alpha, 0)
    
    return img, heatmap_colored, overlay

# ==================== Save Individual Images ====================
def save_individual_images(img_path, heatmap, prediction, confidence, image_id, class_label):
    """Save 3 separate images + 1 combined horizontal view"""
    # Create overlay and get components
    original, heatmap_colored, overlay = create_overlay(img_path, heatmap)
    
    # Determine filename prefix
    label_str = "yes" if class_label == "yes" else "no"
    pred_str = "Tumor" if prediction == 1 else "No Tumor"
    
    # Save original image
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(original)
    ax.set_title(f'Original Image', fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    original_path = f"image{image_id}_{label_str}_original.png"
    plt.savefig(original_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    # Save heatmap
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(heatmap_colored)
    ax.set_title(f'Grad-CAM Heatmap', fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    heatmap_path = f"image{image_id}_{label_str}_heatmap.png"
    plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    # Save overlay with prediction
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(overlay)
    ax.set_title(f'Overlay\nPrediction: {pred_str}\nConfidence: {confidence:.2%}', 
                 fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    overlay_path = f"image{image_id}_{label_str}_overlay.png"
    plt.savefig(overlay_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    # Save combined horizontal view (all 3 in one figure)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    axes[0].imshow(original)
    axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(heatmap_colored)
    axes[1].set_title('Grad-CAM Heatmap', fontsize=14, fontweight='bold')
    axes[1].axis('off')
    
    axes[2].imshow(overlay)
    axes[2].set_title(f'Overlay\nPrediction: {pred_str}\nConfidence: {confidence:.2%}', 
                      fontsize=14, fontweight='bold')
    axes[2].axis('off')
    
    plt.tight_layout()
    combined_path = f"image{image_id}_{label_str}_combined.png"
    plt.savefig(combined_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  ✓ Saved: {original_path}, {heatmap_path}, {overlay_path}, {combined_path}")
    
    return pred_str, confidence

# ==================== Process Test Images ====================
def process_gradcam_analysis(test_dir, num_samples=3):
    """Generate Grad-CAM visualizations for tumor and no-tumor examples"""
    
    # Initialize Grad-CAM (target last convolutional layer)
    target_layer = resnet50.layer4[-1]  # Last residual block in ResNet50
    gradcam = GradCAM(resnet50, target_layer)
    
    results = []
    
    # Process both classes
    for class_label in ['yes', 'no']:
        class_name = "Tumor" if class_label == 'yes' else "No Tumor"
        print(f"\n{'='*70}")
        print(f"Processing {class_name} Images")
        print(f"{'='*70}")
        
        # Get image paths
        class_dir = Path(test_dir) / class_label
        image_paths = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png'))
        
        if len(image_paths) == 0:
            print(f"⚠ No images found in {class_dir}")
            continue
        
        # Randomly sample images
        random.seed(RANDOM_SEED)
        sampled_paths = random.sample(image_paths, min(num_samples, len(image_paths)))
        
        # Process each sampled image
        for idx, img_path in enumerate(sampled_paths, 1):
            image_id = idx if class_label == 'yes' else idx + 3  # Sequential: 1-3 tumor, 4-6 no tumor
            
            print(f"\n[Image {image_id}] {img_path.name}")
            
            # Load and preprocess image
            img_tensor, _ = load_and_preprocess_image(img_path)
            img_tensor = img_tensor.to(device)
            
            # Get model prediction
            with torch.no_grad():
                output = resnet50(img_tensor)
                probs = F.softmax(output, dim=1)
                predicted_class = output.argmax(dim=1).item()
                confidence = probs[0, predicted_class].item()
            
            # Generate Grad-CAM heatmap
            heatmap, _ = gradcam.generate_heatmap(img_tensor, class_idx=predicted_class)
            
            # Save 3 individual images
            pred_str, conf = save_individual_images(
                img_path, heatmap, predicted_class, confidence, image_id, class_label
            )
            
            # Store results
            results.append({
                'image_id': image_id,
                'filename': img_path.name,
                'true_label': class_name,
                'prediction': pred_str,
                'confidence': conf
            })
    
    return results

# ==================== Execute Grad-CAM Analysis ====================
print(f"\n{'='*70}")
print("Starting Grad-CAM Explainability Analysis - ResNet50")
print(f"{'='*70}")
print(f"Test directory: {TEST_DIR}")
print(f"Processing 3 tumor + 3 no-tumor examples")

results = process_gradcam_analysis(TEST_DIR, num_samples=3)

# ==================== Summary ====================
print(f"\n{'='*70}")
print("ANALYSIS SUMMARY")
print(f"{'='*70}")

for r in results:
    status = "✓" if r['prediction'] == r['true_label'] else "✗"
    print(f"{status} Image {r['image_id']} ({r['true_label']}): Predicted {r['prediction']} with {r['confidence']:.2%} confidence")

print(f"\n{'='*70}")
print("✓ Grad-CAM analysis complete!")
print(f"  Generated {len(results) * 4} images (4 per example: original, heatmap, overlay, combined)")
print(f"  Saved in: {Path.cwd()}")
print(f"{'='*70}")

Using device: cpu

Loading ResNet50 model...
✓ ResNet50 loaded successfully (23,512,130 parameters)

Starting Grad-CAM Explainability Analysis - ResNet50
Test directory: ../dataset_processed/test
Processing 3 tumor + 3 no-tumor examples

Processing Tumor Images

[Image 1] y64.jpg


c:\Users\mohdf\OneDrive - Singapore Institute Of Technology\Capstone\Project\venv\Lib\site-packages\torch\nn\modules\module.py:1866: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


  ✓ Saved: image1_yes_original.png, image1_yes_heatmap.png, image1_yes_overlay.png, image1_yes_combined.png

[Image 2] y1210.jpg
  ✓ Saved: image2_yes_original.png, image2_yes_heatmap.png, image2_yes_overlay.png, image2_yes_combined.png

[Image 3] y1054.jpg
  ✓ Saved: image3_yes_original.png, image3_yes_heatmap.png, image3_yes_overlay.png, image3_yes_combined.png

Processing No Tumor Images

[Image 4] no64.jpg
  ✓ Saved: image4_no_original.png, image4_no_heatmap.png, image4_no_overlay.png, image4_no_combined.png

[Image 5] no1205.jpg
  ✓ Saved: image5_no_original.png, image5_no_heatmap.png, image5_no_overlay.png, image5_no_combined.png

[Image 6] no1048.jpg
  ✓ Saved: image6_no_original.png, image6_no_heatmap.png, image6_no_overlay.png, image6_no_combined.png

ANALYSIS SUMMARY
✓ Image 1 (Tumor): Predicted Tumor with 100.00% confidence
✓ Image 2 (Tumor): Predicted Tumor with 99.99% confidence
✓ Image 3 (Tumor): Predicted Tumor with 99.98% confidence
✓ Image 4 (No Tumor): Predicted No Tu